In [1]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [8]:
import sqlite3
connection = sqlite3.connect("../db.sqlite3")
cursor = connection.cursor()
query = """
SELECT
    restaurant_order.id AS order_id,
    restaurant_order.datetime,
    restaurant_orderitem.quantity,
    restaurant_product.price,
    restaurant_product.name AS product_name
FROM restaurant_order
JOIN restaurant_orderitem ON restaurant_order.id = restaurant_orderitem.order_id
JOIN restaurant_product ON restaurant_orderitem.product_id = restaurant_product.id
"""
cursor.execute(query)
rows = cursor.fetchall()
df = pd.DataFrame(rows, columns=["order_id", "datetime", "quantity", "price", "product_name"])
df.to_csv("restaurant_order.csv", index=False)
cursor.close()
connection.close()
data = pd.read_csv("restaurant_order.csv")
data.head()

,order_id,datetime,quantity,price,product_name
0,1,2019-08-03 20:25:00,2,0.80,Plain Papadum
1,1,2019-08-03 20:25:00,1,12.95,King Prawn Balti
2,1,2019-08-03 20:25:00,1,2.95,Garlic Naan
3,1,2019-08-03 20:25:00,1,3.95,Mushroom Rice
4,1,2019-08-03 20:25:00,1,8.95,Paneer Tikka Masala


# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
import matplotlib.pyplot as plt
top_10_products = data.groupby("product_name")["quantity"].sum().sort_values(ascending=False)[:10]
plt.figure(figsize=(6, 6))
plt.pie(
    top_10_products,
    labels=top_10_products.index,
    autopct="%1.1f%%"
)
plt.title("Top 10 positions in menu by quantity")
plt.tight_layout()
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
data["item_price"] = data["price"] * data["quantity"]
top_10_make_money = data.groupby("product_name")["item_price"].sum().sort_values(ascending=False)[:10]
plt.figure(figsize=(6, 6))
plt.pie(
    top_10_make_money,
    labels=top_10_make_money.index,
    autopct="%1.1f%%"
)
plt.title("Top 10 positions in menu by making money")
plt.tight_layout()
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
data["datetime"] = pd.to_datetime(data["datetime"])
data["item_price"] = data["price"] * data["quantity"]
data["hours"] = data["datetime"].dt.hour
hourly_income = data.groupby("hours")["item_price"].sum()
plt.figure(figsize=(12, 6))
plt.bar(hourly_income.index, hourly_income.values)
plt.title("Profit by Order Hour")
plt.xlabel("Order Hour")
plt.xticks(range(24))
plt.grid(axis="y", alpha=0.2)
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
data["datetime"] = pd.to_datetime(data["datetime"])
data["day_of_week"] = data["datetime"].dt.dayofweek
days = {0: "Monday", 1: "Tuesday", 2: "Wednesday", 3: "Thursday", 4: "Friday", 5: "Saturday", 6: "Sunday"}
data["day_name"] = data["day_of_week"].map(days)
daily_income = data.groupby(["day_of_week", "day_name"])["item_price"].sum().reset_index().sort_values("day_of_week")

plt.figure(figsize=(12, 6))
plt.bar(daily_income["day_name"], daily_income["item_price"])
plt.title("Profit by Day of Week")
plt.xlabel("Day of Week")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()